# Locally quadratic asymptotically linear 

For the lunar rover, note that the reference tracking cost is currently chaotic.
The problem is that the $z\text{-axis}$ linear acceleration reference can never truly match the small lunar gravity.
So, if we use a quadratic cost function, then the $z\text{-axis}$ is given emphasis in the solution approaches to the nonlinear solvers.
(E.g., a small change in $z$ gives a much larger change in the cost than a moderate change in $x$ or $y$.)

To resolve this scale on the derivative, note that a linear cost is desired.
Namely, this would enforce similar scaling for the $x$, $y$, and $z$ references, regardless of large offsets, e.g., caused by small lunar gravity.
We run into some problems.

1. Linear costs are not locally differentiable.
(This can cause problems if we are really good at our references tracking, which is our goal.)

2. Linear costs are not rotationally symmetric.

>Note that SMS also said something about how cost functions should be symmetric about what angle the table is pointing in the $xy\text{-plane}$, which is something that I've never really thought about.


For (1), we introduce the namesake: locally quadratic asymptotically linear (LQAL).
Essentially, squaring a norm makes a norm differentiable at zero (at least in the Euclidean context).
So, how about we stitch the quadratic function with a linear function?
A hyperbola also looks good?

For (2), we should probably ask about where the rotational symmetry is most important.
We propose a decomposition.
Let $\phi$ denote LQAL cost function, let $\overrightarrow{\omega} = (\omega_x, \omega_y, \omega_z) \in \mathbb{R}^3$ denote an angular velocity, and let $\overrightarrow{a} = (a_x, a_y, a_z) \in \mathbb{R}^3$ denote an angular acceleration.

The original cost function is of the form
$$
  \|\Delta \overrightarrow{\omega}\|^2 + \|\Delta \overrightarrow{a}\|^2 = (\Delta \omega_x)^2 + (\Delta \omega_y)^2 + (\Delta \omega_z)^2 + (\Delta a_x)^2 + (\Delta a_y)^2 + (\Delta a_z)^2,
$$
where $\Delta$ is the operator that computes the difference with the reference values.
Instead, we propose
$$
  \phi(\|\Delta \overrightarrow{\omega}\|) + \phi(\|\Delta (a_x, a_y)\|) + \Phi(|\Delta a_z|).
$$
This modification lacks an azimuthal angular symmetry in the linear acceleration, but the gradient information is much better captured.
This is considered a desirable trade off.


In [ ]:
%matplotlib ipympl

import matplotlib.pyplot as plt
import numpy as np
import jax
import jax.numpy as jnp
import sympy as sp
import functools
import typing as tp

## Visualizing some solutions to problem (1)

In [ ]:
def phi0(a: float, b: float, x: jax.Array) -> jax.Array:
    # return b * jnp.sqrt(1 + jnp.square(x**2 / a)) - b
    # return b * jnp.sqrt(jnp.sqrt(1 + jnp.square(x**2 / a))) - b
    return b * jnp.sqrt(1 + jnp.square(x / a)) - b

In [ ]:
_hyper = functools.partial(phi0, 2**0, 2**0)

fig, ax = plt.subplots()
xs = jnp.linspace(-10.0, 10.0, 2**10)
plt.plot(xs, jax.vmap(_hyper)(xs), label=r'$hyper$')
plt.show()

### Visualizing the algebra

Honestly, this looks pretty good.
I think that I want to implement this.
If needed, I will consider the LQAL cost.
But a hyperbolic cost function just looks _so_ natural.

In [ ]:
x = sp.Symbol("x")
y = sp.Symbol("y")
z = sp.Symbol("z")
a = sp.Symbol("a")
b = sp.Symbol("b")
norm = sp.sqrt(x**2 + y**2 + z**2)
# norm = sp.sqrt(x)
phi0_sym = b * sp.sqrt(1 + (norm / a)**2)
# phi0_sym = b * sp.sqrt(sp.sqrt(1 + (norm**2 / a)**2))
phi0_sym_diff = sp.diff(phi0_sym, x, 1)
phi0_sym_diff_simplified = sp.simplify(phi0_sym_diff)
phi0_sym_diff_simplified